In [1]:
from pathlib import Path
import os
import json
from datetime import datetime

In [2]:
from main_train import get_args
from utils.config import load_config
from data.dataset import get_vqav2
from data.dataset import get_filtered_trainval
from data.text_processing import save_tokenizer

from data.custom_generators import get_custom_generators
from models.vqa_models import get_model

from train.trainer import train_with_KD, train_from_scratch

from data.onehot_encoder import OneHotEncoder

In [3]:
config = load_config()

In [4]:
##### Per mio computer ############################################################
config["paths"] = {
    "dataset_path": Path("/Users/glori/Desktop/STMicroelectronics/VQA improved/vqa_dataset"),
    "glove_path": Path("/Users/glori/Desktop/STMicroelectronics/prove VQA torch/glove.6B"),
    "KD_path":  Path("/Users/glori/Desktop/STMicroelectronics/VQA KD"),
    "output_path": Path("outputs")
}

In [5]:
saving_folder = Path("outputs") / "MFBBaseline_250807_1723"
config["paths"]["saving_folder"] = saving_folder

In [6]:
config["model"]["num_classes"] = 992
config["model"]

{'model_architecture': 'MFBBaseline',
 'max_length': 15,
 'num_vocab_words': -1,
 'min_frequency': 5,
 'image_size': 224,
 'num_channels': 3,
 'num_classes': 992,
 'consider_teacher': True,
 'k_window': 5,
 'output_MFB': 1024,
 'num_attention_glimps': 2,
 'embedding_dim': 100,
 'use_glove': True,
 'dropout_rate': 0}

In [7]:
split = "train2014"

In [8]:
df = get_vqav2(
    config["paths"]["dataset_path"], split=split, keep_10ans=True, verbose=True
)

Set:  train2014
Total number of sample in train2014: 443757


In [9]:
model_ans_path = config["paths"]["saving_folder"] / f"{split}_answers.json"

In [10]:
##### Per mio computer ############################################################
train_im = os.listdir(config['paths']['dataset_path']/'train2014')

df = df[df['image_name'].isin(train_im)]

len(df)

10754

In [11]:
from main_eval import prepare_and_generate_answers

In [12]:
# NOTAAAA: ne ho considerate solo 50 perché non volevo ci mettesse troppo, cmq va.

In [13]:
model_ans_s = prepare_and_generate_answers(df, config)

Inference: 100%|██████████████████████████████| 337/337 [02:50<00:00,  1.98it/s]


In [14]:
model_ans_s[:10]

[{'question_id': 9000, 'answer': 'broccoli'},
 {'question_id': 9001, 'answer': 'broccoli'},
 {'question_id': 9002, 'answer': 'broccoli'},
 {'question_id': 25000, 'answer': 'log'},
 {'question_id': 25001, 'answer': 'log'},
 {'question_id': 25002, 'answer': 'log'},
 {'question_id': 25003, 'answer': 'log'},
 {'question_id': 25004, 'answer': 'log'},
 {'question_id': 25005, 'answer': 'log'},
 {'question_id': 25006, 'answer': 'log'}]

In [15]:
with open(model_ans_path, "w") as file:
    json.dump(model_ans_s, file)

In [16]:
import pandas as pd

In [17]:
df_model_ans = pd.DataFrame(model_ans_s)
df_model_ans.head(3)

,question_id,answer
0,9000,broccoli
1,9001,broccoli
2,9002,broccoli


In [18]:
df = pd.merge(df, df_model_ans, how="outer", on="question_id")
df.head(5)

,question_type,multiple_choice_answer,answer_type,question_id,normalized_answer,normalized_10answers,image_id,question,image_name,answer
0,how many,2,number,9000,2,"[2, 2, 2, 2, 2, 2, 2, 2, 2, 2]",9,How many cookies can be seen?,COCO_train2014_000000000009.jpg,broccoli
1,what color are the,pink and yellow,other,9001,pink and yellow,"[pink and yellow, yellow pink, pink yellow blu...",9,What color are the dishes?,COCO_train2014_000000000009.jpg,broccoli
2,what is the,broccoli,other,9002,broccoli,"[broccoli, broccoli, broccoli, broccoli, brocc...",9,What is the green stuff?,COCO_train2014_000000000009.jpg,broccoli
3,what is,tree,other,25000,tree,"[tree, tree, tree, tree, tree, tree, tree, tre...",25,What is in front of the giraffes?,COCO_train2014_000000000025.jpg,log
4,what,eating,other,25001,eating,"[height, long neck, spots, they are tall, eati...",25,What do these giraffes have in common?,COCO_train2014_000000000025.jpg,log


In [19]:
from train.performance import vqa_accuracy

In [20]:
ten_ans = list(df["normalized_10answers"])
model_ans = list(df["answer"])
accuracy_all = vqa_accuracy(model_ans, ten_ans)
performance = {"all": accuracy_all}

In [21]:
performance

{'all': 0.13006013266381516}

In [22]:
for ans_type in ["yes/no", "number", "other"]:
    sub_df = df[df["answer_type"] == ans_type]
    ten_ans = list(sub_df["normalized_10answers"])
    model_ans = list(sub_df["answer"])
    performance[ans_type] = vqa_accuracy(model_ans, ten_ans)

In [23]:
performance

{'all': 0.13006013266381516,
 'yes/no': 0.0,
 'number': 0.03547169811320755,
 'other': 0.22774501544509992}

In [24]:
with open(config["paths"]["saving_folder"] / f"{split}_accuracy.json", "w") as file:
    json.dump({split: performance}, file)